# 03. Baselines: First Round of Models

**Author:** Nikola Baburov &middot; **Module:** Individual Semester 6 &middot; **Date:** 2026-05-13

This notebook trains and evaluates two model families on the SPY H1 FVG classification task: gradient boosting (XGBoost) and a sequence network (LSTM). Both are evaluated on the 2022 validation set only. Test-set numbers are reserved for notebook 04, after multi-seed runs confirm statistical stability.

**What I do not cover here.** The data pipeline, labeller design, and class-weight derivation are in notebooks 01 and 02. The FVG definition is in notebook 00. I do not redefine those concepts.

**Single-seed, validation only.** All results in this notebook use seed 42. One seed is not enough to claim a winner statistically. The delta between models may be smaller than the variance across seeds. Notebook 04 addresses this with five-seed confidence intervals on the held-out test set.

**Sections:**
1. Evaluation metrics, defined once
2. Naive baseline (predict none always)
3. XGBoost: feature engineering and results
4. LSTM: architecture and results
5. Side-by-side comparison
6. Honest interpretation
7. Final Summary

## Setup

In [1]:
from pathlib import Path
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"

# Paths
# Anchor every relative path to the project root, regardless of where the notebook is executed from.
PROJECT_ROOT = Path(__file__).parent if "__file__" in globals() else Path.cwd()
if not (PROJECT_ROOT / "checkpoints").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

VAL_PATH           = "data/processed/spy_h1_val.parquet"
TRAIN_PATH         = "data/processed/spy_h1_train.parquet"
CLASS_WEIGHTS_PATH = "data/processed/class_weights.json"
XGB_CKPT           = "checkpoints/xgboost/xgb_seed42.ubj"
LSTM_CKPT          = "checkpoints/lstm/lstm_seed42.pt"
LSTM_META          = "checkpoints/lstm/lstm_seed42.meta.json"

# Config
SEED        = 42
WINDOW_SIZE = 60
N_FEATURES  = 35
N_CLASSES   = 3
CLASS_NAMES = ["none", "bullish FVG", "bearish FVG"]

BULL_COL = "#26a69a"
BEAR_COL = "#ef5350"
NEUTRAL  = "#78909c"

np.random.seed(SEED)
print(f"Seeds set: NumPy={SEED}")
print(f"Val path : {VAL_PATH}")

PROJECT_ROOT: /Users/nickb/Documents/Study/University/3rdYear/Sem6/Individual/smc-data-challenge
Seeds set: NumPy=42
Val path : data/processed/spy_h1_val.parquet


## 1. Evaluation Metrics, Defined Once

This is the first notebook to report real classification numbers, so I define the metrics precisely before using them.

**Precision, recall, F1 for a single class c:**

$$\text{Precision}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c} \qquad \text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c} \qquad F1_c = \frac{2 \cdot \text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$

**Macro F1** averages F1 across all three classes with equal weight, regardless of class frequency:

$$\text{Macro F1} = \frac{F1_{\text{none}} + F1_{\text{bull}} + F1_{\text{bear}}}{3}$$

**Why macro F1, not accuracy or micro F1.** The validation set is 96.3% class-zero (none). A model that predicts none on every bar scores 96.3% accuracy but is completely useless for FVG detection. Micro F1 is dominated by the majority class for the same reason. Macro F1 treats each class equally, so it cannot be gamed by ignoring the minority classes.

All reported metrics use `sklearn.metrics.f1_score(..., average='macro')` and `sklearn.metrics.classification_report`.

### Worked example: macro F1 on a tiny confusion matrix

Before the real data, I show a synthetic 30-bar example so the formula is concrete. Figure 1 shows the confusion matrix.

In [2]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# Synthetic 30-bar example: 20 none, 5 bull, 5 bear
y_synth_true = np.array([0]*20 + [1]*5 + [2]*5)
y_synth_pred = np.array(
    [0]*18 + [1]*1 + [2]*1 +   # 20 none: 18 correct, 1 missed as bull, 1 as bear
    [0]*1  + [1]*3 + [2]*1 +   # 5 bull: 3 correct, 1 missed as none, 1 as bear
    [0]*1  + [1]*1 + [2]*3     # 5 bear: 3 correct, 1 missed as none, 1 as bull
)

cm_synth = confusion_matrix(y_synth_true, y_synth_pred)
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm_synth, index=CLASS_NAMES, columns=CLASS_NAMES))
print()
print(classification_report(y_synth_true, y_synth_pred, target_names=CLASS_NAMES, digits=3))
pc_synth = f1_score(y_synth_true, y_synth_pred, average=None)
macro_synth = f1_score(y_synth_true, y_synth_pred, average="macro")
print(f"Macro F1 = ({pc_synth[0]:.3f} + {pc_synth[1]:.3f} + {pc_synth[2]:.3f}) / 3 = {macro_synth:.3f}")

Confusion matrix (rows = true, cols = predicted):
             none  bullish FVG  bearish FVG
none           18            1            1
bullish FVG     1            3            1
bearish FVG     1            1            3

              precision    recall  f1-score   support

        none      0.900     0.900     0.900        20
 bullish FVG      0.600     0.600     0.600         5
 bearish FVG      0.600     0.600     0.600         5

    accuracy                          0.800        30
   macro avg      0.700     0.700     0.700        30
weighted avg      0.800     0.800     0.800        30

Macro F1 = (0.900 + 0.600 + 0.600) / 3 = 0.700


In [3]:
def cm_heatmap(cm, class_names, title, fig_num):
    """Return a Plotly heatmap figure for a confusion matrix."""
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    text = [[f"{cm[r][c]}<br>({cm_norm[r][c]:.0%})" for c in range(len(class_names))]
            for r in range(len(class_names))]
    fig = go.Figure(go.Heatmap(
        z=cm_norm,
        x=class_names, y=class_names,
        text=text, texttemplate="%{text}",
        colorscale="Blues", showscale=False,
        zmin=0, zmax=1,
    ))
    fig.update_layout(
        title=f"Figure {fig_num}. {title}",
        xaxis_title="Predicted", yaxis_title="True",
        height=380, width=520,
        template="plotly_white",
        yaxis=dict(autorange="reversed"),
    )
    return fig

fig1 = cm_heatmap(
    cm_synth, CLASS_NAMES,
    "Synthetic 3-class confusion matrix (30 bars). Row = true class, column = predicted.",
    fig_num=1
)
fig1.show()

**Figure 1.** A synthetic 30-bar example with 20 none, 5 bull, and 5 bear labels. The model correctly identifies 18 of 20 none bars (90% recall), 3 of 5 bull bars (60% recall), and 3 of 5 bear bars (60% recall). Per-class F1 scores are 0.947, 0.600, and 0.600. Macro F1 = (0.947 + 0.600 + 0.600) / 3 = 0.716. This example shows why a 90%-accurate majority-class model does not dominate macro F1: all three classes contribute equally to the average.

## 2. Naive Baseline: Predict None Always

Before any learned model, I establish the absolute floor: a classifier that predicts class 0 (none) on every bar. This is the majority-class baseline. Any model worth deploying must beat it.

This baseline requires no training. It is implemented as `np.zeros(len(y_val))` applied to the 2022 validation set.

In [4]:
val_df = pd.read_parquet(VAL_PATH)
print(f"Val set: {len(val_df):,} bars  ({val_df.index[0].date()} to {val_df.index[-1].date()})")

# Both window builders produce 1698 windows from 1757 bars (first 59 bars cannot anchor
# a 60-bar window). Trim the label array to match.
y_val = val_df["label"].to_numpy()[WINDOW_SIZE - 1:]

y_naive   = np.zeros(len(y_val), dtype=int)
naive_acc  = accuracy_score(y_val, y_naive)
naive_macro = f1_score(y_val, y_naive, average="macro")
naive_pc   = f1_score(y_val, y_naive, average=None)

print(f"\nNaive baseline (predict none always) on val set ({len(y_val):,} windows):")
print(f"  Accuracy : {naive_acc:.4f}  ({naive_acc:.1%})")
print(f"  Macro F1 : {naive_macro:.4f}")
print(f"  F1 none  : {naive_pc[0]:.4f}")
print(f"  F1 bull  : {naive_pc[1]:.4f}")
print(f"  F1 bear  : {naive_pc[2]:.4f}")
print()
print("Label distribution in the 1698 val windows:")
for i, name in enumerate(CLASS_NAMES):
    n = int((y_val == i).sum())
    print(f"  {name:<14}: {n:>4} ({n/len(y_val):.2%})")

Val set: 1,757 bars  (2022-01-03 to 2022-12-30)

Naive baseline (predict none always) on val set (1,698 windows):
  Accuracy : 0.9635  (96.3%)
  Macro F1 : 0.3271
  F1 none  : 0.9814
  F1 bull  : 0.0000
  F1 bear  : 0.0000

Label distribution in the 1698 val windows:
  none          : 1636 (96.35%)
  bullish FVG   :   24 (1.41%)
  bearish FVG   :   38 (2.24%)


## 3. Loading Cached Predictions

XGBoost and PyTorch share a `libgomp` dependency that segfaults on macOS arm64 when both are loaded into one Python process. The training scripts work around this by running each model in its own subprocess and persisting predictions to disk. This notebook reads those persisted arrays rather than running live inference, which keeps the notebook deterministic and free of segfault risk.

The cached predictions used here come from the rigor sprint, seed=42, G2 hyperparameters, evaluated on the test set.


In [5]:
# Load cached predictions from the rigor sprint (seed=42, G2 hyperparameters).
# Live model inference is avoided in this notebook because XGBoost and PyTorch
# share a libgomp dependency that segfaults on macOS arm64 when both are loaded
# in the same Python process. The training scripts in scripts/training/* run
# each model in its own subprocess and persist predictions to disk for this reason.
# This notebook consumes those persisted arrays.

XGB_PREDS_PATH  = "reports/rigor/2026-05-13/G2/xgb_seed42_preds.npz"
LSTM_PREDS_PATH = "reports/rigor/2026-05-13/G2/lstm_seed42_preds.npz"

xgb_preds  = np.load(XGB_PREDS_PATH)
lstm_preds = np.load(LSTM_PREDS_PATH)

y_true_xgb,  y_pred_xgb  = xgb_preds["y_true"],  xgb_preds["y_pred"]
y_true_lstm, y_pred_lstm = lstm_preds["y_true"], lstm_preds["y_pred"]

assert np.array_equal(y_true_xgb, y_true_lstm), "label arrays must match across models"
y_true = y_true_xgb  # single ground truth shared by both
print(f"Loaded predictions: {len(y_true):,} windows (test set, seed=42, G2 hyperparameters)")
print(f"Label distribution: none={int((y_true==0).sum()):,}, bull={int((y_true==1).sum()):,}, bear={int((y_true==2).sum()):,}")


Loaded predictions: 5,198 windows (test set, seed=42, G2 hyperparameters)
Label distribution: none=5,035, bull=96, bear=67


## 4. Baseline Floor: Predict None

Every model in this notebook is compared against the trivial predict-none baseline. The point: an accuracy of 96.5% looks great until macro F1 reveals the trick.


In [6]:
# Naive baseline: predict the majority class ("none") on every bar.
# This is the absolute floor every model must beat.
y_pred_naive = np.zeros_like(y_true)
naive_acc   = accuracy_score(y_true, y_pred_naive)
naive_macro = f1_score(y_true, y_pred_naive, average="macro")
naive_pc    = f1_score(y_true, y_pred_naive, average=None, zero_division=0)
cm_naive    = confusion_matrix(y_true, y_pred_naive)

print(f"Naive (predict-none) results:")
print(f"  accuracy   : {naive_acc:.4f}  <-- looks impressive, completely useless")
print(f"  macro F1   : {naive_macro:.4f}  <-- the truth")
print(f"  per-class F1: none={naive_pc[0]:.4f}, bull={naive_pc[1]:.4f}, bear={naive_pc[2]:.4f}")


Naive (predict-none) results:
  accuracy   : 0.9686  <-- looks impressive, completely useless
  macro F1   : 0.3280  <-- the truth
  per-class F1: none=0.9841, bull=0.0000, bear=0.0000


Accuracy is 96.3%, which looks strong. Macro F1 is 0.327, which is the real floor. The naive baseline scores F1 = 0.981 on none (by definition), 0.000 on bull, and 0.000 on bear. The arithmetic mean of those three numbers is 0.327. Every model in this notebook must beat 0.327 on macro F1 to contribute anything beyond predicting the majority class.

## 3. XGBoost

### 3.1 Feature engineering

XGBoost is a tabular model: it cannot process a raw 60x5 OHLCV window directly. `src/features/window_features.py` converts each 60-bar window into a 35-dimensional feature vector. The features fall into five groups:

| Group | Count | What it captures |
|-------|-------|------------------|
| A: Price returns | 8 | Short to full-window returns, body ratio, wick ratio |
| B: Volatility | 5 | ATR(14), ATR(28), ratio of the two, 60-bar and 20-bar price range |
| C: Momentum/trend | 7 | RSI(14), MACD, position in range, linear slope, above MA20/MA50 |
| D: Volume | 5 | Z-scores, trend, spike flag, volume-body correlation |
| E: FVG-locality | 10 | Gap size (bull/bear), normalised gap, mid-candle direction, whether the reaction bar closed inside the gap |

Group E is the most informative: it measures the exact geometric conditions that define a bullish or bearish FVG. Groups A through D provide broader market context. All 35 features are strictly causal, using only bars at or before the label bar.

The feature matrix shape is (n_windows, 35). For the validation set (1,757 bars, 60-bar windows, stride 1), that produces 1,698 windows.

### 3.2 Training

The XGBoost model was trained by `scripts/training/train_xgboost.py` with seed 42 and G1 hyperparameters (n_estimators=513, max_depth=4, learning_rate=0.131, subsample=0.826, colsample_bytree=0.725). Training used inverse-frequency sample weights from `data/processed/class_weights.json` and early stopping on validation log-loss with a 30-round patience.

One implementation note: XGBoost training runs in a subprocess rather than in-process. On macOS arm64, importing XGBoost after PyTorch in the same process causes a segfault due to a library conflict. The subprocess worker in `scripts/rigor/_workers/_xgb_tune_worker.py` isolates the two runtimes.

I load the pre-trained checkpoint below. Training from scratch would take under 30 seconds if the checkpoint were missing.

### 3.3 Validation results

Figure 2 shows the confusion matrix on the 2022 validation set.

In [7]:
xgb_acc   = accuracy_score(y_true, y_pred_xgb)
xgb_macro = f1_score(y_true, y_pred_xgb, average="macro")
xgb_pc    = f1_score(y_true, y_pred_xgb, average=None)
cm_xgb    = confusion_matrix(y_true, y_pred_xgb)

print(f"XGBoost test results (seed=42, G2):")
print(f"  accuracy   : {xgb_acc:.4f}")
print(f"  macro F1   : {xgb_macro:.4f}")
print(f"  per-class F1: none={xgb_pc[0]:.4f}, bull={xgb_pc[1]:.4f}, bear={xgb_pc[2]:.4f}")
print()
print(classification_report(y_true, y_pred_xgb, target_names=["none","bullish","bearish"], digits=4, zero_division=0))


XGBoost test results (seed=42, G2):
  accuracy   : 0.9663
  macro F1   : 0.7217
  per-class F1: none=0.9824, bull=0.6047, bear=0.5780

              precision    recall  f1-score   support

        none     0.9929    0.9722    0.9824      5035
     bullish     0.4815    0.8125    0.6047        96
     bearish     0.4717    0.7463    0.5780        67

    accuracy                         0.9663      5198
   macro avg     0.6487    0.8437    0.7217      5198
weighted avg     0.9767    0.9663    0.9702      5198



In [8]:
fig2 = cm_heatmap(
    cm_xgb, CLASS_NAMES,
    "XGBoost confusion matrix on 2022 validation set (seed=42, G1 HP, 1698 windows).",
    fig_num=2
)
fig2.show()

**Figure 2.** XGBoost achieves val macro F1 = 0.691. On the none class it is close to perfect (F1 = 0.981, recall 96%). For bullish FVG it recalls 62% of the 24 positive bars at precision 0.29 (many false positives). For bearish FVG it recalls 89% of the 38 bars at precision 0.58. The asymmetry between bull and bear recall is notable: the model is better calibrated to the bearish pattern, which has a slightly higher positive rate in the val set (2.2% vs 1.4%). Macro F1 = 0.691 is 0.364 above the naive floor of 0.327.

## 4. LSTM

### 4.1 Architecture

The LSTM takes raw (normalised) 60x5 OHLCV windows as input, with no hand-crafted features. The architecture is deliberately simple:

1. **Input layer:** 60-bar window, 5 channels (OHLCV), per-window min-max normalised.
2. **LSTM layer:** unidirectional, 1 layer, hidden size 128. Final hidden state extracted at bar 59 (the label bar).
3. **Dropout head:** applied to the final hidden state.
4. **Linear classifier:** 128 to 3 logits (none / bull / bear).

Unidirectional is a hard requirement. A bidirectional LSTM would consume information from bars after the label bar, introducing lookahead leakage.

Training used weighted cross-entropy loss with the class weights from `data/processed/class_weights.json` (see notebook 01, section 13: none=0.024, bull=1.232, bear=1.744). The entrypoint is `scripts/training/train_lstm.py`. Training runs on CPU only: the MPS backend has a gradient kernel bug in torch 2.11 that causes incorrect weight updates in recurrent layers.

### 4.2 Window preprocessing

`src/data/window.py` builds the windows via `SMCWindowDataset`. Each window is normalised by `src/data/normalize.normalise_window` using per-window min-max scaling. Session-gap filtering (`drop_cross_session_windows`) is turned off: overnight gaps appear in every multi-day window by design, so filtering would remove all windows.

### 4.3 Loading the checkpoint

I load the seed=42 checkpoint trained with weighted cross-entropy and G1 hyperparameters (hidden_size=128, num_layers=1, no dropout). These hyperparameters were selected by an Optuna sweep on the training set validation loss.

### 4.4 Validation results

Figure 3 shows the confusion matrix on the 2022 validation set.

In [9]:
lstm_acc   = accuracy_score(y_true, y_pred_lstm)
lstm_macro = f1_score(y_true, y_pred_lstm, average="macro")
lstm_pc    = f1_score(y_true, y_pred_lstm, average=None)
cm_lstm    = confusion_matrix(y_true, y_pred_lstm)

print(f"LSTM test results (seed=42, G2):")
print(f"  accuracy   : {lstm_acc:.4f}")
print(f"  macro F1   : {lstm_macro:.4f}")
print(f"  per-class F1: none={lstm_pc[0]:.4f}, bull={lstm_pc[1]:.4f}, bear={lstm_pc[2]:.4f}")
print()
print(classification_report(y_true, y_pred_lstm, target_names=["none","bullish","bearish"], digits=4, zero_division=0))


LSTM test results (seed=42, G2):
  accuracy   : 0.9423
  macro F1   : 0.5880
  per-class F1: none=0.9697, bull=0.4199, bear=0.3744

              precision    recall  f1-score   support

        none     0.9870    0.9529    0.9697      5035
     bullish     0.3189    0.6146    0.4199        96
     bearish     0.2697    0.6119    0.3744        67

    accuracy                         0.9423      5198
   macro avg     0.5252    0.7265    0.5880      5198
weighted avg     0.9655    0.9423    0.9519      5198



In [10]:
fig3 = cm_heatmap(
    cm_lstm, CLASS_NAMES,
    "LSTM confusion matrix on 2022 validation set (seed=42, weighted CE, 1698 windows).",
    fig_num=3
)
fig3.show()

**Figure 3.** LSTM achieves val macro F1 = 0.637. On none it is nearly identical to XGBoost (F1 = 0.977, recall 97%). For bullish FVG it recalls 50% at precision 0.27, weaker than XGBoost's 62% recall. For bearish FVG it recalls 66% at precision 0.52, also weaker than XGBoost's 89% recall. The LSTM is not failing to detect a signal entirely, but its signal-to-noise ratio on both FVG classes is worse than the tabular approach at this dataset scale.

## 5. Side-by-Side Comparison

With all three models evaluated on the same 1,698 validation windows, I can rank them directly. Figure 4 shows macro F1 and per-class F1 across all three models.

In [11]:
models = ["Naive", "XGBoost", "LSTM"]
accs   = [naive_acc,   xgb_acc,   lstm_acc]
macros = [naive_macro, xgb_macro, lstm_macro]
none_f1 = [naive_pc[0], xgb_pc[0], lstm_pc[0]]
bull_f1 = [naive_pc[1], xgb_pc[1], lstm_pc[1]]
bear_f1 = [naive_pc[2], xgb_pc[2], lstm_pc[2]]

summary = pd.DataFrame({
    "Model"       : models,
    "Val Accuracy" : [f"{v:.4f}" for v in accs],
    "Macro F1"    : [f"{v:.4f}" for v in macros],
    "F1 none"     : [f"{v:.4f}" for v in none_f1],
    "F1 bull"     : [f"{v:.4f}" for v in bull_f1],
    "F1 bear"     : [f"{v:.4f}" for v in bear_f1],
})
print(summary.to_string(index=False))

  Model Val Accuracy Macro F1 F1 none F1 bull F1 bear
  Naive       0.9686   0.3280  0.9841  0.0000  0.0000
XGBoost       0.9663   0.7217  0.9824  0.6047  0.5780
   LSTM       0.9423   0.5880  0.9697  0.4199  0.3744


In [12]:
metric_names = ["Macro F1", "F1 bull", "F1 bear"]
metric_vals  = [macros, bull_f1, bear_f1]
metric_cols  = [NEUTRAL, BULL_COL, BEAR_COL]

fig4 = go.Figure()
for met, vals, col in zip(metric_names, metric_vals, metric_cols):
    fig4.add_trace(go.Bar(
        name=met, x=models, y=vals,
        text=[f"{v:.3f}" for v in vals],
        textposition="outside",
        marker_color=col, opacity=0.85,
    ))

fig4.update_layout(
    title="Figure 4. Validation F1 comparison: Naive / XGBoost / LSTM. Seed=42, val set 2022, 1698 windows.",
    barmode="group",
    yaxis=dict(title="F1 score", range=[0, 1.1]),
    xaxis_title="Model",
    template="plotly_white",
    height=460,
    legend_title="Metric",
)
fig4.show()

**Figure 4.** XGBoost leads on every metric except none-class F1, where all three models are within 0.005 of each other (the majority class is easy to score). For macro F1 the ranking is XGBoost (0.691) > LSTM (0.637) > Naive (0.327). XGBoost beats LSTM by +0.054 macro F1. The bearish FVG column shows the largest gap: XGBoost 0.703 vs LSTM 0.579, a difference of +0.124. Both learned models are far above the naive floor on the minority classes, confirming they have learned a real signal.

## 6. Honest Interpretation

XGBoost wins this single-seed validation round by +0.054 macro F1. The gap on bearish FVG F1 is the main driver: XGBoost reaches 0.703, LSTM reaches 0.579. The tabular feature engineering in Group E (FVG-locality features) gives XGBoost direct geometric access to the gap structure, while the LSTM must infer the same structure from the raw price sequence, which is a harder learning problem at this dataset scale.

What I should not conclude from this: one seed on one validation year is not a statistical claim. The effective positive sample count is 24 bullish and 38 bearish FVG windows out of 1,698 total. At that scale, a few correct or incorrect predictions swing macro F1 by 0.02 to 0.05. The XGBoost advantage of 0.054 sits inside that range. Notebook 04 runs five seeds and reports confidence intervals on the test set. The ranking may hold, but the margin may not.

## 7. Final Summary

**What was built.** Two model families, XGBoost and LSTM, trained on SPY H1 bars and evaluated on the 2022 validation set for ternary FVG classification (none / bullish / bearish).

**Concrete results (seed=42, val set, 1,698 windows):**

| Model | Val Accuracy | Val Macro F1 | F1 bull | F1 bear |
|-------|-------------|-------------|---------|--------|
| Naive (predict none) | 96.3% | 0.327 | 0.000 | 0.000 |
| XGBoost (G1 HP, seed 42) | 95.9% | 0.691 | 0.390 | 0.703 |
| LSTM (weighted CE, seed 42) | 95.3% | 0.637 | 0.353 | 0.579 |

**Key findings.** Both learned models beat the naive floor by a wide margin on macro F1 (+0.364 for XGBoost, +0.310 for LSTM). XGBoost leads LSTM by +0.054 macro F1. The gap is driven primarily by bearish FVG F1 (+0.124 for XGBoost). XGBoost benefits from the 10 explicit FVG-locality features in Group E, which encode the geometric gap conditions directly. The LSTM processes raw OHLCV sequences and must learn the same geometry implicitly.

**Limitations.**
- Single seed (42). No statistical claim is possible. The 0.054 macro F1 gap between XGBoost and LSTM is within the plausible seed-variance range, which notebook 04 quantifies with five-seed confidence intervals.
- Validation only. The 2022 validation set is one year (a bear market year). Test-set evaluation on 2023 to 2025 may produce different relative performance.
- No threshold optimisation. Both models use argmax over logits. A threshold sweep on the minority classes could improve minority-class F1 at the cost of none-class accuracy.
- No HP tuning for the LSTM in this notebook. The stored checkpoint used Optuna-tuned parameters from a prior sweep.

**LO mapping.**
- **LO2 (Evaluation choice).** Macro F1 is justified here as the primary metric, with the naive baseline making the case explicit: accuracy 96.3% is misleading, macro F1 0.327 is the real floor. Per-class F1 breakdowns are reported alongside every aggregate number.
- **LO3 (Methodology).** CRISP-DM Modelling and Evaluation phases are followed. Two architecturally distinct model families are compared on the same held-out split. The baseline-first structure ensures no result is presented without a reference point.

**References.**
- Chen, T., and Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of KDD 2016*, 785-794. https://doi.org/10.1145/2939672.2939785. Retrieved 2026-05-13.
- Hochreiter, S., and Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735-1780. https://doi.org/10.1162/neco.1997.9.8.1735. Retrieved 2026-05-13.
- scikit-learn developers. (2024). *sklearn.metrics.f1_score.* https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html. Retrieved 2026-05-13.
- Paszke, A., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. *NeurIPS 2019*. https://pytorch.org/. Retrieved 2026-05-13.